# PPE Safety Detection — YOLOv6 vs YOLOv7 vs YOLOv8 vs YOLOv10 on 3 splits

Trains and tests **4 models × 3 splits (60/20/20, 70/15/15, 80/10/10) = 12 runs** on the Roboflow
`ppe-detection-qlq3d-ivtcj` dataset (10 classes) and prints train logs, test metrics (P, R, F1, mAP@50, mAP@50-95),
per-class AP and a final comparison table.

**How to run**
1. `Runtime → Change runtime type → T4 GPU`. Add your Roboflow key as a Colab secret named `ROBOFLOW_API_KEY`.
2. Run **Steps 1–5** (setup + data). Then run **one model section per session** (Step 6 = YOLOv6, 7 = YOLOv7, 8 = YOLOv8, 9 = YOLOv10).
   Every finished run is saved to Drive and skipped on re-run, so a disconnected session just resumes.
3. Run **Step 10** for the combined results. Every run also saves its test detections (`test_predictions.csv`) so the separate *compliance* notebook can score helmet + vest compliance without a GPU.

Tip: set `SMOKE_TEST = True` first (1 epoch, one split, roughly 15-20 min for all models) to catch any environment problem before the long runs.

**Fair-comparison choices:** same data, 640 px, batch 16, 50 epochs, seed 42, COCO-pretrained nano-class weights
(YOLOv6n, YOLOv7-tiny, YOLOv8n, YOLOv10n), no early stopping, model selected on the *validation* split, metrics reported on the *test* split.

In [ ]:
# ============================================================
# STEP 1 — CONFIG  (edit only this cell)
# ============================================================
import os, sys, json, time, shutil, re, math, gc, subprocess, traceback, warnings
from pathlib import Path
import numpy as np
import pandas as pd

SMOKE_TEST = False            # True = 1 epoch, first split only (dry run)

EPOCHS   = 1 if SMOKE_TEST else 50
IMG_SIZE = 640
BATCH    = 16
SEED     = 42
WORKERS  = 2                  # Colab has 2 vCPUs
PRED_MIN_CONF = 0.01          # lowest confidence kept when saving test detections (used by the compliance analysis)

# Models to run in THIS session. Use "YOLOv7" instead of "YOLOv7-tiny" for the full-size model (~3x slower).
RUN_MODELS = ["YOLOv6n", "YOLOv7-tiny", "YOLOv8n", "YOLOv10n"]

# split name -> (train %, val %, test %)
SPLIT_RATIOS = {
    "split_60_20_20": (60, 20, 20),
    "split_70_15_15": (70, 15, 15),
    "split_80_10_10": (80, 10, 10),
}
if SMOKE_TEST:
    SPLIT_RATIOS = {"split_60_20_20": (60, 20, 20)}

# Roboflow dataset
RF_WORKSPACE, RF_PROJECT, RF_VERSION = "shivu-k-s", "ppe-detection-qlq3d-ivtcj", 1

# Paths
RAW_DIR       = Path("/content/PPE-detection-1")                 # Roboflow download
SPLIT_ROOT    = Path("/content/ppe_splits")                      # fast local copy of the 3 splits
DRIVE_PROJECT = Path("/content/drive/MyDrive/PPE_Safety_Detection")
RUNS_ROOT     = DRIVE_PROJECT / ("final_runs_smoke" if SMOKE_TEST else "final_runs")

FAILED = []                   # (model, split, error) collected during the runs

print("Smoke test :", SMOKE_TEST)
print("Epochs     :", EPOCHS, "| img", IMG_SIZE, "| batch", BATCH, "| seed", SEED)
print("Models     :", RUN_MODELS)
print("Splits     :", list(SPLIT_RATIOS))
print("Results dir:", RUNS_ROOT)

In [ ]:
# ============================================================
# STEP 2 — GPU CHECK + ENVIRONMENT FLAGS
# ============================================================
import torch

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Use Runtime -> Change runtime type -> T4 GPU, then re-run.")
print("GPU     :", torch.cuda.get_device_name(0))

# Disable W&B (it blocks YOLOv7 asking for a login) and allow loading old pickled checkpoints (torch >= 2.6).
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

In [ ]:
# ============================================================
# STEP 3 — INSTALL PACKAGES   (estimated runtime: 1-2 min)
# ============================================================
def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

pip("ultralytics", "roboflow", "pycocotools", "thop", "addict", "scikit-learn", "tensorboard", "seaborn", "scipy")

import ultralytics
print("Ultralytics:", ultralytics.__version__)

In [ ]:
# ============================================================
# STEP 4 — GOOGLE DRIVE + ROBOFLOW DOWNLOAD   (estimated runtime: 1 min)
# ============================================================
import yaml
from google.colab import drive

drive.mount("/content/drive")
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

if not (RAW_DIR / "data.yaml").exists():
    from google.colab import userdata
    from roboflow import Roboflow
    try:
        rf_key = userdata.get("ROBOFLOW_API_KEY")
    except Exception:
        raise RuntimeError("Add your Roboflow API key as a Colab secret named ROBOFLOW_API_KEY "
                           "(key icon in the left sidebar) and enable notebook access.")
    rf = Roboflow(api_key=rf_key)
    rf.workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION).download("yolov8", location=str(RAW_DIR))
else:
    print("Dataset already downloaded:", RAW_DIR)

raw_cfg = yaml.safe_load((RAW_DIR / "data.yaml").read_text())
names = raw_cfg["names"]
CLASS_NAMES = [names[i] for i in sorted(names)] if isinstance(names, dict) else list(names)
NC = len(CLASS_NAMES)
print("\nClasses (%d): %s" % (NC, CLASS_NAMES))

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
for part in ["train", "valid", "test"]:
    n_img = sum(1 for p in (RAW_DIR / part / "images").iterdir() if p.suffix.lower() in IMG_EXTS)
    n_lab = len(list((RAW_DIR / part / "labels").glob("*.txt")))
    print(f"original {part:5s}: {n_img} images | {n_lab} label files")

In [ ]:
# ============================================================
# STEP 5 — BUILD THE 3 SPLITS (clean labels, leak-free, class-stratified)
# Estimated runtime: 2-4 min (the image copy runs once per runtime)
# ============================================================
from collections import Counter
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings("ignore", message=".*least populated class.*")

def source_key(stem):
    # Roboflow names augmented copies "<source>_jpg.rf.<hash>". All copies of one source image share a key.
    return stem.split(".rf.")[0]

def clean_label(path, nc, stats):
    # Return clean 'cls cx cy w h' rows: polygons -> boxes, boxes clipped to [0,1], degenerate boxes dropped.
    rows = []
    if not path.exists():
        return rows
    for line in path.read_text().splitlines():
        p = line.split()
        if not p:
            continue
        cls = int(float(p[0]))
        vals = [float(v) for v in p[1:]]
        if not 0 <= cls < nc:
            raise ValueError(f"{path.name}: class id {cls} outside 0..{nc-1}")
        if len(vals) == 4:
            cx, cy, w, h = vals
        elif len(vals) >= 6 and len(vals) % 2 == 0:
            xs, ys = vals[0::2], vals[1::2]
            x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
            cx, cy, w, h = (x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1
            stats["polygon_rows"] += 1
        else:
            stats["malformed_rows"] += 1
            continue
        box = (cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)
        clipped = tuple(min(1.0, max(0.0, v)) for v in box)
        if any(abs(a - b) > 1e-6 for a, b in zip(box, clipped)):
            stats["clipped_rows"] += 1
        bw, bh = clipped[2] - clipped[0], clipped[3] - clipped[1]
        if bw <= 1e-6 or bh <= 1e-6:
            stats["degenerate_rows"] += 1
            continue
        rows.append(f"{cls} {(clipped[0]+clipped[2])/2:.6f} {(clipped[1]+clipped[3])/2:.6f} {bw:.6f} {bh:.6f}")
    return rows

# ---- 1. collect every image of the Roboflow download (train + valid + test pooled) ----
stats = Counter()
records = []
for part in ["train", "valid", "test"]:
    for img in sorted((RAW_DIR / part / "images").iterdir()):
        if img.suffix.lower() not in IMG_EXTS:
            continue
        rows = clean_label(RAW_DIR / part / "labels" / f"{img.stem}.txt", NC, stats)
        records.append({"img": img, "rows": rows, "key": source_key(img.stem),
                        "classes": sorted({int(r.split()[0]) for r in rows})})
assert len({r["img"].name for r in records}) == len(records), "duplicate file names across original splits"

groups = np.array([r["key"] for r in records])
print("Total images        :", len(records))
print("Unique source groups:", len(set(groups)), "(augmented copies of one photo share a group)")
print("Label fixes         :", dict(stats))

# ---- 2. stratification label = rarest class present in the image ----
freq = Counter(c for r in records for c in r["classes"])
y = np.array([min(r["classes"], key=lambda c: freq[c]) if r["classes"] else -1 for r in records])

def make_folds(k):
    skf = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=SEED)
    fold_of = np.full(len(records), -1)
    for f, (_, te) in enumerate(skf.split(np.zeros(len(records)), y, groups)):
        fold_of[te] = f
    return fold_of

# ---- 3. one split per ratio: k folds of 100/gcd % each -> test folds, val folds, rest = train ----
marker = SPLIT_ROOT / ".built_ok.json"
signature = {"ratios": SPLIT_RATIOS, "seed": SEED, "n": len(records)}
already_built = marker.exists() and json.loads(marker.read_text()) == json.loads(json.dumps(signature))

PPE_IDS = {n.replace("-", "_"): CLASS_NAMES.index(n) for n in ["helmet", "vest", "no-helmet", "no-vest"] if n in CLASS_NAMES}
if len(PPE_IDS) < 4:
    print("WARNING: helmet / vest / no-helmet / no-vest not all found in CLASS_NAMES -> compliance ground truth skipped")
gt_rows = []
summary, manifest = [], []
for split, (tr, va, te) in SPLIT_RATIOS.items():
    assert tr + va + te == 100
    g = math.gcd(math.gcd(tr, va), te)
    k = 100 // g
    fold_of = make_folds(k)
    test_f = set(range(0, te // g))
    val_f = set(range(te // g, te // g + va // g))
    part_of = np.array(["test" if f in test_f else "valid" if f in val_f else "train" for f in fold_of])

    # no source photo may appear in two parts
    key_parts = {}
    for key, part in zip(groups, part_of):
        key_parts.setdefault(key, set()).add(part)
    assert all(len(v) == 1 for v in key_parts.values()), "leak: a source image sits in two parts"

    root = SPLIT_ROOT / split
    if not already_built:
        if root.exists():
            shutil.rmtree(root)
        for part in ["train", "valid", "test"]:
            (root / part / "images").mkdir(parents=True)
            (root / part / "labels").mkdir(parents=True)
        for rec, part in zip(records, part_of):
            shutil.copy2(rec["img"], root / part / "images" / rec["img"].name)
            (root / part / "labels" / f"{rec['img'].stem}.txt").write_text("\n".join(rec["rows"]) + ("\n" if rec["rows"] else ""))

    cfg = {"train": str(root / "train/images"), "val": str(root / "valid/images"), "test": str(root / "test/images"),
           "nc": NC, "names": CLASS_NAMES}
    (root / "data.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))
    # YOLOv6 only regenerates its COCO ground truth for the 'val' task -> evaluate TEST by pointing 'val' at test images
    (root / "data_test_as_val.yaml").write_text(yaml.safe_dump(dict(cfg, val=cfg["test"]), sort_keys=False))

    for part in ["train", "valid", "test"]:
        idx = np.where(part_of == part)[0]
        inst = Counter(int(r.split()[0]) for i in idx for r in records[i]["rows"])
        summary.append({"split": split, "part": part, "images": len(idx),
                        "pct": round(100 * len(idx) / len(records), 1),
                        "groups": len({groups[i] for i in idx}), "boxes": sum(inst.values()),
                        **{CLASS_NAMES[c]: inst.get(c, 0) for c in range(NC)}})
    if len(PPE_IDS) == 4:                      # per-image PPE box counts of the TEST part (for the compliance analysis)
        for i in np.where(part_of == "test")[0]:
            cnt = Counter(int(r.split()[0]) for r in records[i]["rows"])
            gt_rows.append({"split": split, "image": records[i]["img"].stem, **{n: cnt.get(ci, 0) for n, ci in PPE_IDS.items()}})
    for rec, part in zip(records, part_of):
        manifest.append({"split": split, "part": part, "image": rec["img"].name, "source_group": rec["key"]})

if not already_built:
    SPLIT_ROOT.mkdir(parents=True, exist_ok=True)
    marker.write_text(json.dumps(signature))

pd.DataFrame(manifest).to_csv(RUNS_ROOT / "split_manifest.csv", index=False)
if gt_rows:
    pd.DataFrame(gt_rows).to_csv(RUNS_ROOT / "test_ground_truth.csv", index=False)
    print("Compliance ground truth saved to:", RUNS_ROOT / "test_ground_truth.csv")
summary_df = pd.DataFrame(summary)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print("\nSPLIT SUMMARY (images, source groups, boxes and boxes per class)")
print(summary_df.to_string(index=False))
print("\nLeak check passed: no source photo appears in more than one of train/valid/test.")
print("Split manifest saved to:", RUNS_ROOT / "split_manifest.csv")

In [ ]:
# ============================================================
# COMMON HELPERS (training launcher, cache purge, metric parsing, reports)
# ============================================================
_PROGRESS = re.compile(r"\b\d{1,2}%\|")      # in-progress tqdm lines (0-99%) are hidden to keep the output readable

def header(title):
    print("\n" + "=" * 72 + f"\n{title}\n" + "=" * 72)

def purge_caches(root):
    # YOLOv7 and Ultralytics both write 'labels.cache' in DIFFERENT formats -> a stale one crashes the other. Always clear.
    root = Path(root)
    for p in list(root.rglob("*.cache")) + list(root.rglob(".images_cache.json")):
        p.unlink(missing_ok=True)
    shutil.rmtree(root / "annotations", ignore_errors=True)

def run_logged(cmd, cwd, log_path, env_extra=None, show=True):
    env = os.environ.copy()
    env.update({"PYTHONWARNINGS": "ignore", "PYTHONUNBUFFERED": "1"})
    env["PYTHONPATH"] = str(cwd) + os.pathsep + env.get("PYTHONPATH", "")
    if env_extra:
        env.update(env_extra)
    Path(log_path).parent.mkdir(parents=True, exist_ok=True)
    lines = []
    with open(log_path, "w") as log:
        proc = subprocess.Popen([str(c) for c in cmd], cwd=str(cwd), env=env, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            log.write(line); lines.append(line)
            if show and not _PROGRESS.search(line):
                print(line, end="", flush=True)
        rc = proc.wait()
    return rc, "".join(lines)

def f1_score(p, r):
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

_NUM = r"[-+]?(?:\d+\.?\d*|\.\d+)(?:[eE][-+]?\d+)?|nan"
def parse_row(text, label, n_floats):
    # Find the LAST table row "<label>  <images> <labels> <n floats>" in a framework's evaluation printout.
    pat = rf"(?<![\w-]){re.escape(label)}\s+(\d+)\s+(\d+)" + "".join(rf"\s+({_NUM})" for _ in range(n_floats))
    found = list(re.finditer(pat, text))
    return [float(x) for x in found[-1].groups()[2:]] if found else None

def count_test_images(split):
    return sum(1 for p in (SPLIT_ROOT / split / "test" / "images").iterdir() if p.suffix.lower() in IMG_EXTS)

def save_metrics(model_key, split, run_dir, overall, per_class):
    d = {"model": model_key, "split": split, "epochs": EPOCHS, "test_images": count_test_images(split),
         **overall, "per_class": per_class}
    (Path(run_dir) / "metrics.json").write_text(json.dumps(d, indent=2))
    return d

def report(d):
    print(f"\n{d['model']} | {d['split']} | TEST SET ({d['test_images']} images) | trained {d['epochs']} epochs")
    print(f"  Precision : {d['precision']:.4f}")
    print(f"  Recall    : {d['recall']:.4f}")
    print(f"  F1        : {d['f1']:.4f}")
    print(f"  mAP@50    : {d['map50']:.4f}")
    print(f"  mAP@50-95 : {d['map50_95']:.4f}")
    df = pd.DataFrame(d["per_class"]).T[["precision", "recall", "ap50", "ap50_95"]]
    df.columns = ["P", "R", "mAP@50", "mAP@50-95"]
    print("\n  Per-class:")
    print(df.astype(float).round(4).to_string())

def write_predictions(run_dir, rows):
    # rows = (image stem, class name, confidence). Saved so the compliance notebook can run without a GPU.
    df = pd.DataFrame(rows, columns=["image", "cls_name", "conf"])
    df = df[df["conf"] >= PRED_MIN_CONF]
    if df.empty:
        msg = "No test detections were parsed - the compliance analysis would be meaningless (check test_*.log)."
        if SMOKE_TEST:
            print("WARNING:", msg)
        else:
            raise RuntimeError(msg)
    df.to_csv(Path(run_dir) / "test_predictions.csv", index=False)
    print(f"Saved {len(df)} test detections ({df['image'].nunique()} images) -> test_predictions.csv")

def read_txt_predictions(label_root):
    # YOLO txt files with 'cls x y w h conf' per line -> (image stem, class name, conf)
    rows = []
    for f in Path(label_root).rglob("*.txt"):
        for ln in f.read_text().split("\n"):
            p = ln.split()
            if len(p) >= 6:
                rows.append((f.stem, CLASS_NAMES[int(float(p[0]))], float(p[5])))
    return rows

def run_complete(run_dir):
    return (Path(run_dir) / "metrics.json").exists() and (Path(run_dir) / "test_predictions.csv").exists()

def release_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def guarded(model_key, split, fn):
    # One failing run must not kill the remaining runs; the error is printed and listed at the end.
    try:
        fn()
    except Exception as e:
        traceback.print_exc()
        FAILED.append((model_key, split, f"{type(e).__name__}: {e}"))
        print(f"\n!! {model_key} / {split} FAILED -> continuing with the next run")
    finally:
        release_gpu()

def patch_text(path, replacements):
    # Idempotent in-place patching of third-party repo files. Returns number of replacements applied.
    path = Path(path)
    text = path.read_text()
    n = 0
    for old, new in replacements:
        if old in text:                      # 'old' never occurs inside 'new', so a second run changes nothing
            n += text.count(old)
            text = text.replace(old, new)
    path.write_text(text)
    return n

def add_weights_only_false(path, calls):
    # torch >= 2.6 defaults to weights_only=True which rejects these old pickled checkpoints.
    return patch_text(path, [(c, c[:-1] + ", weights_only=False)") for c in calls])

print("Helpers ready.")

## Step 6 — YOLOv6n (Meituan repo, COCO-pretrained `yolov6n.pt`)
Fixes versus the earlier attempt: the repo config trains from scratch (`pretrained=None`) — here it fine-tunes the COCO weights; and
test evaluation is run with the test images mapped to YOLOv6's `val` task, because YOLOv6 only regenerates its COCO ground-truth JSON for `val`
(that stale-JSON reuse caused `Results do not correspond to current coco set`).

In [ ]:
# ============================================================
# STEP 6 — YOLOv6n: 3 splits (train -> test)
# Estimated runtime: roughly 30-60 min per split on a T4 (this is a guess; check the first epochs)
# ============================================================
V6 = Path("/content/YOLOv6")

def setup_yolov6():
    if not V6.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/meituan/YOLOv6.git", str(V6)], check=True)
    n = add_weights_only_false(V6 / "yolov6/utils/checkpoint.py", [
        "torch.load(weights, map_location=map_location)",
        "torch.load(ckpt_path, map_location=torch.device('cpu'))"])
    n += add_weights_only_false(V6 / "yolov6/core/engine.py", [
        "torch.load(args.resume, map_location='cpu')",
        "torch.load(weights, map_location=device)"])
    w = V6 / "weights" / "yolov6n.pt"
    w.parent.mkdir(exist_ok=True)
    if not w.exists():
        import urllib.request
        urllib.request.urlretrieve("https://github.com/meituan/YOLOv6/releases/download/0.4.0/yolov6n.pt", w)
    cfg = (V6 / "configs/yolov6n.py").read_text()
    assert "pretrained=None" in cfg, "unexpected YOLOv6 config layout"
    (V6 / "configs/yolov6n_ppe.py").write_text(cfg.replace("pretrained=None", f"pretrained='{w}'", 1))
    print(f"YOLOv6 ready (torch.load patches applied: {n}) | pretrained weights: {w} ({w.stat().st_size/1e6:.1f} MB)")

def parse_yolov6_eval(text):
    allrow = parse_row(text, "all", 5)                      # P, R, F1, mAP50, mAP50-95 (at best-F1 point)
    ap_all = re.search(r"Average Precision\s+\(AP\) @\[ IoU=0\.50:0\.95 \| area=\s*all \| maxDets=100 \] = ([\d.]+)", text)
    ap_50  = re.search(r"Average Precision\s+\(AP\) @\[ IoU=0\.50\s+\| area=\s*all \| maxDets=100 \] = ([\d.]+)", text)
    if allrow is None and not (ap_all and ap_50):
        raise RuntimeError("Could not find metrics in the YOLOv6 evaluation output (see test/eval.log).")
    p, r = (allrow[0], allrow[1]) if allrow else (float("nan"), float("nan"))
    overall = {"precision": p, "recall": r, "f1": f1_score(p, r) if allrow else float("nan"),
               "map50": float(ap_50.group(1)) if ap_50 else allrow[3],
               "map50_95": float(ap_all.group(1)) if ap_all else allrow[4]}
    per_class = {}
    for name in CLASS_NAMES:
        row = parse_row(text, name, 5)
        per_class[name] = ({"precision": row[0], "recall": row[1], "ap50": row[3], "ap50_95": row[4]} if row else
                           {"precision": float("nan"), "recall": float("nan"), "ap50": float("nan"), "ap50_95": float("nan")})
    return overall, per_class

def one_yolov6_run(split):
    model_key = "YOLOv6n"
    run_dir = RUNS_ROOT / model_key / split
    run_dir.mkdir(parents=True, exist_ok=True)
    header(f"{model_key} | {split}")
    if run_complete(run_dir):
        print("Already finished - saved metrics:")
        report(json.loads((run_dir / "metrics.json").read_text()))
        return
    root = SPLIT_ROOT / split
    purge_caches(root)

    if not (run_dir / "train_done.txt").exists():
        for d in run_dir.glob("train*"):
            shutil.rmtree(d, ignore_errors=True)            # restart unfinished attempt from scratch
        print("--- TRAINING ---")
        rc, _ = run_logged([sys.executable, V6 / "tools/train.py",
                            "--data-path", root / "data.yaml", "--conf-file", V6 / "configs/yolov6n_ppe.py",
                            "--img-size", IMG_SIZE, "--batch-size", BATCH, "--epochs", EPOCHS,
                            "--workers", WORKERS, "--device", "0", "--eval-interval", 1,
                            "--output-dir", run_dir, "--name", "train"], V6, run_dir / "train.log")
        if rc != 0:
            raise RuntimeError(f"YOLOv6 training exited with code {rc} (see {run_dir/'train.log'})")
        (run_dir / "train_done.txt").write_text(time.strftime("%Y-%m-%d %H:%M:%S"))

    best = sorted(run_dir.glob("train*/weights/best_ckpt.pt"), key=lambda p: p.stat().st_mtime)[-1]
    print("\n--- TEST EVALUATION ---\nBest checkpoint:", best)
    purge_caches(root)
    rc, text = run_logged([sys.executable, V6 / "tools/eval.py",
                           "--data", root / "data_test_as_val.yaml", "--weights", best, "--task", "val",
                           "--device", "0", "--batch-size", BATCH, "--img-size", IMG_SIZE,
                           "--conf-thres", 0.001, "--iou-thres", 0.65, "--verbose",
                           "--save_dir", run_dir / "test", "--name", "eval"], V6, run_dir / "test_eval.log")
    if rc != 0:
        raise RuntimeError(f"YOLOv6 evaluation exited with code {rc} (see {run_dir/'test_eval.log'})")
    overall, per_class = parse_yolov6_eval(text)
    report(save_metrics(model_key, split, run_dir, overall, per_class))

    # per-image detections for the compliance analysis (eval.py cannot save them -> use infer.py)
    raw = run_dir / "test_predictions_raw"
    shutil.rmtree(raw, ignore_errors=True)
    (raw / "labels").mkdir(parents=True); (raw / "images" / "labels").mkdir(parents=True)   # infer.py nests its output differently depending on the source path
    rc, _ = run_logged([sys.executable, V6 / "tools/infer.py", "--weights", best, "--source", root / "test/images",
                        "--yaml", root / "data.yaml", "--img-size", IMG_SIZE, IMG_SIZE, "--conf-thres", PRED_MIN_CONF,
                        "--iou-thres", 0.65, "--max-det", 300, "--device", "0", "--save-txt", "--not-save-img",
                        "--save-dir", raw], V6, run_dir / "test_predict.log", show=False)
    if rc != 0:
        raise RuntimeError(f"YOLOv6 prediction export exited with code {rc} (see {run_dir/'test_predict.log'})")
    write_predictions(run_dir, read_txt_predictions(raw))

if "YOLOv6n" in RUN_MODELS:
    setup_yolov6()
    for _split in SPLIT_RATIOS:
        guarded("YOLOv6n", _split, lambda s=_split: one_yolov6_run(s))

## Step 7 — YOLOv7-tiny (WongKinYiu repo, COCO-pretrained `yolov7-tiny.pt`)
Fixes versus the earlier attempt: stale/foreign `labels.cache` files are deleted before every run (this was the `unpickling stack underflow` crash),
W&B is disabled, and the repo is patched for torch ≥ 2.6 / NumPy 2 (`weights_only`, loss-index dtype, `np.int`, `np.trapz`).

In [ ]:
# ============================================================
# STEP 7 — YOLOv7-tiny: 3 splits (train -> test)
# Estimated runtime: roughly 30-60 min per split on a T4 (this is a guess; check the first epochs)
# ============================================================
V7 = Path("/content/yolov7")
V7_VARIANTS = {
    "YOLOv7-tiny": dict(weights="yolov7-tiny.pt", cfg="cfg/training/yolov7-tiny.yaml",
                        hyp="data/hyp.scratch.tiny.yaml", batch=BATCH),
    "YOLOv7":      dict(weights="yolov7_training.pt", cfg="cfg/training/yolov7.yaml",
                        hyp="data/hyp.scratch.p5.yaml", batch=8),      # full model: batch 8 to fit a T4
}

def setup_yolov7(variant):
    if not V7.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/WongKinYiu/yolov7.git", str(V7)], check=True)
    n = add_weights_only_false(V7 / "train.py", ["torch.load(weights, map_location=device)"])
    n += add_weights_only_false(V7 / "models/experimental.py", ["torch.load(w, map_location=map_location)"])
    n += add_weights_only_false(V7 / "utils/datasets.py", ["torch.load(cache_path)"])
    n += add_weights_only_false(V7 / "utils/general.py", ["torch.load(f, map_location=torch.device('cpu'))"])
    n += patch_text(V7 / "utils/datasets.py", [(".astype(np.int)", ".astype(int)")])
    n += patch_text(V7 / "utils/loss.py", [("gj.clamp_(0, gain[3] - 1), gi.clamp_(0, gain[2] - 1)",
                                             "gj.clamp_(0, (gain[3] - 1).long()), gi.clamp_(0, (gain[2] - 1).long())")])
    if hasattr(np, "trapezoid"):
        n += patch_text(V7 / "utils/metrics.py", [("np.trapz(", "np.trapezoid(")])
    v = V7_VARIANTS[variant]
    w = V7 / v["weights"]
    if not w.exists():
        import urllib.request
        urllib.request.urlretrieve(f"https://github.com/WongKinYiu/yolov7/releases/download/v0.1/{v['weights']}", w)
    print(f"YOLOv7 ready (source patches applied: {n}) | pretrained weights: {w} ({w.stat().st_size/1e6:.1f} MB)")

def parse_yolov7_eval(text):
    allrow = parse_row(text, "all", 4)                      # P, R, mAP@.5, mAP@.5:.95
    if allrow is None:
        raise RuntimeError("Could not find the 'all' metrics row in the YOLOv7 test output (see test_eval.log).")
    overall = {"precision": allrow[0], "recall": allrow[1], "f1": f1_score(allrow[0], allrow[1]),
               "map50": allrow[2], "map50_95": allrow[3]}
    per_class = {}
    for name in CLASS_NAMES:
        row = parse_row(text, name, 4)
        per_class[name] = ({"precision": row[0], "recall": row[1], "ap50": row[2], "ap50_95": row[3]} if row else
                           {"precision": float("nan"), "recall": float("nan"), "ap50": float("nan"), "ap50_95": float("nan")})
    return overall, per_class

def one_yolov7_run(model_key, split):
    v = V7_VARIANTS[model_key]
    run_dir = RUNS_ROOT / model_key / split
    run_dir.mkdir(parents=True, exist_ok=True)
    header(f"{model_key} | {split}")
    if run_complete(run_dir):
        print("Already finished - saved metrics:")
        report(json.loads((run_dir / "metrics.json").read_text()))
        return
    root = SPLIT_ROOT / split
    purge_caches(root)

    if not (run_dir / "train_done.txt").exists():
        shutil.rmtree(run_dir / "train", ignore_errors=True)  # restart unfinished attempt from scratch
        print("--- TRAINING ---")
        rc, _ = run_logged([sys.executable, "train.py", "--weights", V7 / v["weights"], "--cfg", v["cfg"],
                            "--data", root / "data.yaml", "--hyp", v["hyp"], "--epochs", EPOCHS,
                            "--batch-size", v["batch"], "--img-size", IMG_SIZE, IMG_SIZE, "--device", "0",
                            "--workers", WORKERS, "--project", run_dir, "--name", "train", "--exist-ok"],
                           V7, run_dir / "train.log")
        if rc != 0:
            raise RuntimeError(f"YOLOv7 training exited with code {rc} (see {run_dir/'train.log'})")
        (run_dir / "train_done.txt").write_text(time.strftime("%Y-%m-%d %H:%M:%S"))

    best = run_dir / "train" / "weights" / "best.pt"
    print("\n--- TEST EVALUATION ---\nBest checkpoint:", best)
    shutil.rmtree(run_dir / "test", ignore_errors=True)      # test.py appends to its txt files -> start clean
    rc, text = run_logged([sys.executable, "test.py", "--data", root / "data.yaml", "--weights", best,
                           "--task", "test", "--batch-size", BATCH, "--img-size", IMG_SIZE,
                           "--conf-thres", 0.001, "--iou-thres", 0.65, "--device", "0", "--no-trace",
                           "--verbose", "--save-txt", "--save-conf", "--project", run_dir, "--name", "test", "--exist-ok"],
                          V7, run_dir / "test_eval.log")
    if rc != 0:
        raise RuntimeError(f"YOLOv7 test exited with code {rc} (see {run_dir/'test_eval.log'})")
    overall, per_class = parse_yolov7_eval(text)
    report(save_metrics(model_key, split, run_dir, overall, per_class))
    write_predictions(run_dir, read_txt_predictions(run_dir / "test" / "labels"))

for _key in [m for m in RUN_MODELS if m in V7_VARIANTS]:
    setup_yolov7(_key)
    for _split in SPLIT_RATIOS:
        guarded(_key, _split, lambda k=_key, s=_split: one_yolov7_run(k, s))

## Step 8 — YOLOv8n (Ultralytics)
The same runner is reused for YOLOv10n in Step 9. An interrupted run resumes from `last.pt`.

In [ ]:
# ============================================================
# STEP 8 — YOLOv8n: 3 splits (train -> test)
# Estimated runtime: roughly 25-50 min per split on a T4 (this is a guess; check the first epochs)
# ============================================================
def ultralytics_metrics(m, class_names):
    b = m.box
    p, r = float(b.mp), float(b.mr)
    overall = {"precision": p, "recall": r, "f1": f1_score(p, r), "map50": float(b.map50), "map50_95": float(b.map)}
    per_class = {n: {"precision": float("nan"), "recall": float("nan"), "ap50": float("nan"), "ap50_95": float("nan")}
                 for n in class_names}
    for i, ci in enumerate(b.ap_class_index):                # only classes present in the test set have an entry
        pc, rc, ap50, ap = b.class_result(i)
        per_class[class_names[int(ci)]] = {"precision": float(pc), "recall": float(rc),
                                           "ap50": float(ap50), "ap50_95": float(ap)}
    return overall, per_class

def count_rows(csv_path):
    return max(0, sum(1 for _ in open(csv_path)) - 1) if Path(csv_path).exists() else 0

def one_ultralytics_run(model_key, weights_name, split):
    from ultralytics import YOLO
    run_dir = RUNS_ROOT / model_key / split
    run_dir.mkdir(parents=True, exist_ok=True)
    header(f"{model_key} | {split}")
    if run_complete(run_dir):
        print("Already finished - saved metrics:")
        report(json.loads((run_dir / "metrics.json").read_text()))
        return
    root = SPLIT_ROOT / split
    purge_caches(root)
    data = str(root / "data.yaml")
    tdir = run_dir / "train"
    last, best = tdir / "weights" / "last.pt", tdir / "weights" / "best.pt"

    if not (run_dir / "train_done.txt").exists():
        print("--- TRAINING ---")
        if last.exists() and count_rows(tdir / "results.csv") > 0:
            print("Resuming the interrupted run from last.pt ...")
            YOLO(str(last)).train(resume=True)
        else:
            shutil.rmtree(tdir, ignore_errors=True)
            YOLO(weights_name).train(data=data, epochs=EPOCHS, imgsz=IMG_SIZE, batch=BATCH, device=0,
                                     workers=WORKERS, seed=SEED, patience=EPOCHS, pretrained=True,
                                     project=str(run_dir), name="train", exist_ok=True, plots=True, verbose=True)
        (run_dir / "train_done.txt").write_text(time.strftime("%Y-%m-%d %H:%M:%S"))

    print("\n--- TEST EVALUATION ---\nBest checkpoint:", best)
    m = YOLO(str(best)).val(data=data, split="test", imgsz=IMG_SIZE, batch=BATCH, device=0,
                            project=str(run_dir), name="test", exist_ok=True, plots=True, verbose=True)
    overall, per_class = ultralytics_metrics(m, CLASS_NAMES)
    report(save_metrics(model_key, split, run_dir, overall, per_class))

    rows = []                                                # per-image detections for the compliance analysis
    for r in YOLO(str(best)).predict(source=str(root / "test" / "images"), conf=PRED_MIN_CONF, iou=0.65, imgsz=IMG_SIZE,
                                     device=0, stream=True, verbose=False, max_det=300):
        for c, sc in zip(r.boxes.cls.tolist(), r.boxes.conf.tolist()):
            rows.append((Path(r.path).stem, CLASS_NAMES[int(c)], float(sc)))
    write_predictions(run_dir, rows)

if "YOLOv8n" in RUN_MODELS:
    for _split in SPLIT_RATIOS:
        guarded("YOLOv8n", _split, lambda s=_split: one_ultralytics_run("YOLOv8n", "yolov8n.pt", s))

## Step 9 — YOLOv10n (Ultralytics)

In [ ]:
# ============================================================
# STEP 9 — YOLOv10n: 3 splits (train -> test)
# Estimated runtime: roughly 25-50 min per split on a T4 (this is a guess; check the first epochs)
# ============================================================
if "YOLOv10n" in RUN_MODELS:
    for _split in SPLIT_RATIOS:
        guarded("YOLOv10n", _split, lambda s=_split: one_ultralytics_run("YOLOv10n", "yolov10n.pt", s))

## Step 10 — Combined results (reads every finished run from Drive)

In [ ]:
# ============================================================
# STEP 10 — FINAL COMPARISON: 4 models x 3 splits
# ============================================================
import matplotlib.pyplot as plt

MODEL_ORDER = ["YOLOv6n", "YOLOv7-tiny", "YOLOv7", "YOLOv8n", "YOLOv10n"]
SPLIT_ORDER = ["split_60_20_20", "split_70_15_15", "split_80_10_10"]

rows, per_rows = [], []
for f in sorted(RUNS_ROOT.glob("*/*/metrics.json")):
    d = json.loads(f.read_text())
    rows.append({k: d[k] for k in ["model", "split", "epochs", "test_images", "precision", "recall", "f1", "map50", "map50_95"]})
    for cname, v in d["per_class"].items():
        per_rows.append({"model": d["model"], "split": d["split"], "class": cname, **v})

if not rows:
    raise RuntimeError("No finished runs found yet. Run Steps 6-9 first.")

res = pd.DataFrame(rows)
models = [m for m in MODEL_ORDER if m in set(res["model"])]
splits = [s for s in SPLIT_ORDER if s in set(res["split"])]
res.to_csv(RUNS_ROOT / "all_results_summary.csv", index=False)
pd.DataFrame(per_rows).to_csv(RUNS_ROOT / "all_results_per_class.csv", index=False)

header("TEST-SET RESULTS: one table per metric (rows = model, columns = split)")
for metric, title in [("map50", "mAP@50"), ("map50_95", "mAP@50-95"), ("precision", "Precision"),
                      ("recall", "Recall"), ("f1", "F1")]:
    print(f"\n{title}")
    print(res.pivot(index="model", columns="split", values=metric).reindex(index=models, columns=splits).round(4).to_string())

pc = pd.DataFrame(per_rows)
for s in splits:
    header(f"PER-CLASS mAP@50 on the TEST set | {s}")
    print(pc[pc["split"] == s].pivot(index="class", columns="model", values="ap50").reindex(index=CLASS_NAMES, columns=models).round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
width = 0.8 / max(len(splits), 1)
for ax, metric, title in zip(axes, ["map50", "map50_95"], ["mAP@50", "mAP@50-95"]):
    pv = res.pivot(index="model", columns="split", values=metric).reindex(index=models, columns=splits)
    for j, s in enumerate(splits):
        ax.bar(np.arange(len(models)) + j * width, pv[s].values, width, label=s.replace("split_", "").replace("_", "/"))
    ax.set_xticks(np.arange(len(models)) + width * (len(splits) - 1) / 2)
    ax.set_xticklabels(models, rotation=15)
    ax.set_title(f"Test {title}"); ax.set_ylim(0, 1); ax.grid(axis="y", alpha=0.3); ax.legend(title="train/val/test")
plt.tight_layout()
plt.savefig(RUNS_ROOT / "model_comparison.png", dpi=150)
plt.show()

expected = [(m, s) for m in MODEL_ORDER if m in RUN_MODELS or m in models for s in SPLIT_RATIOS]
missing = [(m, s) for m, s in expected if not ((RUNS_ROOT / m / s / "metrics.json").exists())]
print("\nSaved:", RUNS_ROOT / "all_results_summary.csv", "|", RUNS_ROOT / "all_results_per_class.csv", "|", RUNS_ROOT / "model_comparison.png")
if FAILED:
    print("\nRuns that failed in this session:")
    for m, s, e in FAILED:
        print(f"  {m} / {s}: {e}")
if missing:
    print("\nNot finished yet (re-run the matching step; finished runs are skipped):")
    for m, s in missing:
        print(f"  {m} / {s}")
else:
    print("\nAll runs finished.")